# 00 — Setup Sanity Check
**Sprint week: Aug 22 setup block (19:00–22:00)**

Goal: toolchain running. Done = `cache['resid_post', 6].shape` prints and you can explain each dimension.  
No theory today — that's Tuesday's job (ARENA hooks + src/extract.py).

Steps:
1. GPU check
2. Install
3. Repo structure
4. Load model + one generation
5. `run_with_cache` + shape print
6. Explain the shape

---
## Step 1 — GPU check
Runtime → Change runtime type → T4 GPU, then run this.  
If it says 'No devices found', you forgot to set the runtime.

In [ ]:
!nvidia-smi

---
## Step 2 — Install

In [ ]:
!pip install transformer_lens -q

---
## Step 3 — Repo structure

**Before running this cell:** create the GitHub repo `agentic-misalignment-repe` at github.com (public, no template, no README yet).  
Then fill in your GitHub username below and run.

What this does:
- Clones the empty repo into Colab
- Creates the four required directories with `.gitkeep` so git tracks them
- Adds a minimal `README.md`
- Pushes the skeleton

If you'd rather set up git locally instead of in Colab, just create the four folders manually and skip this cell.

In [ ]:
import os

GITHUB_USERNAME = "YOUR_USERNAME"  # <-- fill this in
REPO_NAME = "agentic-misalignment-repe"

# Configure git identity (Colab resets this each session)
!git config --global user.email "you@example.com"
!git config --global user.name "{GITHUB_USERNAME}"

# Clone the empty repo
# Colab will prompt for your GitHub token — use a personal access token (PAT) with repo scope
!git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
%cd {REPO_NAME}

# Create directory structure
for d in ["data", "src", "notebooks", "figures"]:
    os.makedirs(d, exist_ok=True)
    with open(f"{d}/.gitkeep", "w") as f:
        pass

# Minimal README
readme = """# agentic-misalignment-repe

Detecting agentic misalignment via Representation Engineering.  
B.E. Final Year Project — Dept. of Computer Engineering, FCRCE.

## Structure
- `data/` — contrastive pair datasets and extracted activations
- `src/` — reusable extraction, probing, and intervention code
- `notebooks/` — exploratory Colab notebooks
- `figures/` — AUROC plots and other output figures
"""
with open("README.md", "w") as f:
    f.write(readme)

# Commit and push
!git add -A
!git commit -m "chore: scaffold repo structure"
!git push origin main

print("\nDone. Repo structure:")
for root, dirs, files in os.walk("."):
    # skip .git
    dirs[:] = [d for d in dirs if d != ".git"]
    level = root.replace(".", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

---
## Step 4 — Load gpt2-small + one generation

`HookedTransformer` is TransformerLens's wrapper around GPT-2 that adds hooks at every layer —  
that's the mechanism we'll use to read residual-stream activations.  
Today: just confirm it loads and produces English output.

In [ ]:
import torch
from transformer_lens import HookedTransformer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

SEED = 42
torch.manual_seed(SEED)

In [ ]:
model = HookedTransformer.from_pretrained("gpt2", device=device)
model.eval()
print(f"Model loaded. Layers: {model.cfg.n_layers}, d_model: {model.cfg.d_model}")

In [ ]:
# One generation — confirm English output
PROMPT = "The capital of France is"

tokens = model.to_tokens(PROMPT)
with torch.no_grad():
    output = model.generate(tokens, max_new_tokens=5, do_sample=False)

print("Prompt:", PROMPT)
print("Completion:", model.to_string(output[0]))

---
## Step 5 — run_with_cache + shape print

`run_with_cache` runs a forward pass AND caches every intermediate activation.  
`cache['resid_post', L]` gives the residual stream tensor *after* layer L's attention+MLP block.  
This is the substrate our reading vectors will be extracted from.

In [ ]:
with torch.no_grad():
    logits, cache = model.run_with_cache(PROMPT)

# Calendar step 5 — this is the line from the plan
shape = cache["resid_post", 6].shape
print(f"cache['resid_post', 6].shape: {shape}")

---
## Step 6 — Explain the shape

This is the calendar's written deliverable: *"Write what each of [batch, pos, d_model] means."*

In [ ]:
# Inspect the token positions
token_strs = model.to_str_tokens(PROMPT)
print("Tokens:", list(enumerate(token_strs)))
print()

batch, pos, d_model = shape

print(f"batch   = {batch}")
print(f"  -> number of input sequences. 1 because we passed a single string.")
print(f"  -> in extract.py we'll loop pairs and process each prompt separately.")
print()
print(f"pos     = {pos}")
print(f"  -> number of token positions in the input: {token_strs}")
print(f"  -> each position gets its own {d_model}-d residual stream vector.")
print(f"  -> for our reading vectors we'll take the FINAL token position (index -1)")
print(f"  -> because that's where the model 'decides' how to continue the reasoning.")
print()
print(f"d_model = {d_model}")
print(f"  -> width of the residual stream (GPT-2 small is 768).")
print(f"  -> the reading vector we extract will be a direction in this space.")
print(f"  -> diff-of-means: v_L = mean(misaligned_acts) - mean(aligned_acts), shape [{d_model}].")

In [ ]:
# ── DONE CHECK ──────────────────────────────────────────────────────────
# Calendar: "Done = that shape printed and understood."

assert len(shape) == 3, "shape should be 3-dimensional"
assert shape[0] == 1,   "batch should be 1 for a single string input"
assert shape[2] == 768, "d_model should be 768 for GPT-2 small"

print("✓ Setup complete. Toolchain is live.")
print(f"  cache['resid_post', 6].shape = {tuple(shape)}")
print()
print("Next (Sunday 14:00): Read Lynch et al. arXiv:2510.05179 for trace patterns.")
print("Then (Sunday 17:30): Write 15 contrastive pairs into data/pairs.json.")

---
## What's NOT done yet (and when it happens)

| Item | When |
|---|---|
| `resid_pre` vs `mid` vs `post` — what each hook name means | Tue Aug 25 (ARENA cache exercises) |
| Loop over 30 pairs, cache per layer → `[30, 12, 768]` tensor | Tue Aug 25 (write `src/extract.py`) |
| diff-of-means per layer → reading vectors | Wed Aug 26 |
| AUROC figure + streaming halt demo | Wed Aug 26 |

The only deliverable from today is: GPU confirmed, shape printed, done-check passes.